In [1]:
## IMPORTATION DE LA BASE DE DONNÉES ##

import pandas as pd

df = pd.read_csv('df_elec.csv')

print(f"✅ Succès ! {len(df)} molécules chargées.")
display(df.head(10))

✅ Succès ! 534119 molécules chargées.


,Unnamed: 0,smiles,elec_sites,elec_names,MAA_values,elec_GCS_3_cm5,Set
0,0,NOCc1cccc(I)c1,3,double_bond,90.348433,"[-0.01706, 0.12057, -0.11146, -0.08969, 0.0, -...",Train_fold5
1,1,NOCc1cccc(I)c1,4,double_bond,94.924314,"[-0.08969, -0.01706, -0.08484, 0.09939, 0.0, 0...",Train_fold2
2,2,NOCc1cccc(I)c1,5,double_bond,91.330269,"[-0.08484, -0.10499, -0.08969, 0.09122, 0.0, 0...",Train_fold3
3,3,NOCc1cccc(I)c1,6,double_bond,102.683928,"[-0.10499, 0.01492, -0.08484, 0.08707, 0.0, 0....",Train_fold1
4,4,NOCc1cccc(I)c1,7,double_bond,276.204538,"[0.01492, 0.00479, -0.11146, -0.10499, 0.0, 0....",Train_fold3
5,5,NOCc1cccc(I)c1,9,double_bond,106.802353,"[-0.11146, 0.01492, -0.01706, 0.08581, 0.0, 0....",Train_fold2
6,6,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,8,double_bond,59.505283,"[0.72113, -0.68009, -0.74473, -0.73671, 0.0, 0...",Train_fold3
7,7,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,12,double_bond,92.271444,"[-0.03104, 0.09821, 0.05895, -0.07408, 0.0, -0...",Train_fold4
8,8,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,13,double_bond,116.342355,"[-0.07408, -0.03104, -0.10399, 0.10666, 0.0, 0...",Train_fold4
9,9,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,14,double_bond,96.497045,"[-0.10399, 0.0548, -0.07408, 0.10411, 0.0, -0....",Train_fold2


In [2]:
print("🎓 ÉTAPE 1 : Entraînement du Maître (XGBoost)")
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error # 👈 Nouvel import !
import ast

# 1. On prépare les données pour XGBoost (Il lui faut des nombres purs)
# On convertit la colonne des sites en numéros (O1 -> 0, O2 -> 1, etc.)
encoder = LabelEncoder()
df['Site_Encoded'] = encoder.fit_transform(df['elec_sites'])

# On s'assure que les charges sont bien une liste de nombres et pas du texte
def parse_charges(charge_data):
    if isinstance(charge_data, str):
        return ast.literal_eval(charge_data)
    return charge_data

df['charges_list'] = df['elec_GCS_3_cm5'].apply(parse_charges)
# On éclate la liste de charges en plusieurs colonnes distinctes
charges_df = pd.DataFrame(df['charges_list'].tolist(), index=df.index)
charges_df = charges_df.add_prefix('charge_')

# On assemble le tableau final pour le Maître : Site + Toutes les charges
X_master = pd.concat([df[['Site_Encoded']], charges_df], axis=1)
y_master = df['MAA_values']

# 2. Entraînement éclair du Maître
print("🧠 Le Maître étudie les données quantiques...")
# ⚠️ N'oublie pas de remettre tes paramètres Optuna ici si tu les avais modifiés !
xgb_model = xgb.XGBRegressor(n_estimators=1800, max_depth=11, learning_rate=0.0617, min_child_weight=10, subsample=0.840, colsample_bytree=0.723, tree_method='hist', n_jobs=6, random_state=42)
xgb_model.fit(X_master, y_master)

# 3. La Création du Savoir (Distillation)
# Le Maître prédit la MAA pour TOUTE la base de données. 
# Ce sont ces réponses "lissées" que l'Élève devra imiter !
df['Teacher_MAA_Predictions'] = xgb_model.predict(X_master)

# 4. Le Bulletin de Notes du Maître
r2_master = r2_score(y_master, df['Teacher_MAA_Predictions'])
rmse_master = np.sqrt(mean_squared_error(y_master, df['Teacher_MAA_Predictions']))

print("-" * 40)
print("✅ Le Maître a terminé. Voici son niveau réel :")
print(f"📊 Score R²   : {r2_master:.4f}")
print(f"📉 Erreur RMSE : {rmse_master:.2f}")
print("-" * 40)
print(df[['MAA_values', 'Teacher_MAA_Predictions']].head(3))

🎓 ÉTAPE 1 : Entraînement du Maître (XGBoost)
🧠 Le Maître étudie les données quantiques...
----------------------------------------
✅ Le Maître a terminé. Voici son niveau réel :
📊 Score R²   : 0.9746
📉 Erreur RMSE : 11.45
----------------------------------------
   MAA_values  Teacher_MAA_Predictions
0   90.348433                86.821892
1   94.924314                71.193825
2   91.330269                78.447182


In [3]:
print("📝 ÉTAPE 2 : Préparation des textes et Split Académique (Train/Val/Test)")
from datasets import Dataset

# 1. Création du prompt pour l'élève
def build_student_prompt(row):
    return f"SMILES: {row['smiles']} | Site: {row['elec_sites']} | MAA: "

df['student_prompt'] = df.apply(build_student_prompt, axis=1)

# 2. 🔥 Correction Float32 (Indispensable pour le GPU Apple M4)
df['Teacher_MAA_Predictions'] = df['Teacher_MAA_Predictions'].astype('float32')

# 3. 📦 Conversion du DataFrame Pandas en Dataset Hugging Face
# C'est l'étape qui manquait dans ton code !
full_dataset = Dataset.from_pandas(df[['student_prompt', 'Teacher_MAA_Predictions']]).rename_column('Teacher_MAA_Predictions', 'label')

# 4. ✂️ SPLIT ACADÉMIQUE (80% / 10% / 10%)
# Premier split : on sépare le Train (80%) du reste (20%)
train_rest = full_dataset.train_test_split(test_size=0.2, seed=42)

# Deuxième split : on coupe les 20% restants en deux pour avoir Val (10%) et Test (10%)
val_test = train_rest['test'].train_test_split(test_size=0.5, seed=42)

train_dataset = train_rest['train']
val_dataset = val_test['train']
test_dataset = val_test['test']

print(f"📊 Validation du découpage :")
print(f"   - Train (Entraînement) : {len(train_dataset)} molécules")
print(f"   - Val   (Contrôle nuit) : {len(val_dataset)} molécules")
print(f"   - Test  (Examen final)  : {len(test_dataset)} molécules")
print("\n✅ Dataset prêt pour l'étape 3 ! Tu peux lancer la nuit.")



📝 ÉTAPE 2 : Préparation des textes et Split Académique (Train/Val/Test)


/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📊 Validation du découpage :
   - Train (Entraînement) : 427295 molécules
   - Val   (Contrôle nuit) : 53412 molécules
   - Test  (Examen final)  : 53412 molécules

✅ Dataset prêt pour l'étape 3 ! Tu peux lancer la nuit.


In [4]:
import os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import mean_squared_error, r2_score

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("🌙 MODE COMMANDO : Train/Val/Test + Streaming + Métriques Finales")

# 1. Modèle et Tokenizer
model_name = "DeepChem/ChemBERTa-77M-MTR"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

def tokenize_function(examples):
    return tokenizer(examples["student_prompt"], padding="max_length", truncation=True, max_length=128)

# 🌊 Streaming pour sauver la RAM (16Go)
tokenized_train = train_dataset.to_iterable_dataset().map(tokenize_function, batched=True)
tokenized_val = val_dataset.to_iterable_dataset().map(tokenize_function, batched=True)
tokenized_test = test_dataset.to_iterable_dataset().map(tokenize_function, batched=True)

# 2. Configuration Entraînement
training_args = TrainingArguments(
    output_dir="./nuit_perf_finale",
    learning_rate=1e-5,
    per_device_train_batch_size=32, 
    max_steps=60000,               # Environ 5-6 epochs en streaming
    eval_strategy="steps",
    eval_steps=5000,
    save_steps=5000,
    logging_steps=100,
    load_best_model_at_end=False,   # False en streaming pur pour éviter les bugs de buffer
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("🔥 DÉCOLLAGE... La barre de progression arrive !")
trainer.train()

print("\n🏁 EXAMEN FINAL SUR LE TEST SET ISOLÉ...")
# En mode streaming, on récupère les prédictions par itération
all_preds = []
all_labels = []

# On passe le modèle en mode évaluation (important !)
model.eval()
for batch in test_dataset.to_iterable_dataset().map(tokenize_function, batched=True).with_format("torch"):
    with torch.no_grad():
        inputs = {k: v.to(model.device) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
        outputs = model(**inputs)
        all_preds.extend(outputs.logits.cpu().numpy().flatten())
        all_labels.extend(batch['label'].cpu().numpy().flatten())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)

# 📊 CALCUL DES SCORES
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)
relative_rmse = (rmse / np.mean(y_true)) * 100

print("-" * 35)
print(f"✅ R² Score         : {r2:.4f}")
print(f"📉 RMSE             : {rmse:.4f}")
print(f"🎯 RMSE Relatif     : {relative_rmse:.2f}%")
print("-" * 35)

🌙 MODE COMMANDO : Train/Val/Test + Streaming + Métriques Finales


[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `199`.
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 27849.93it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
regression.out_proj.weight | UNEXPECTED | 
regression.out_proj.bias   | UNEXPECTED | 
norm_mean                  | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
norm_std                   | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trainin

🔥 DÉCOLLAGE... La barre de progression arrive !


/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
5000,10043.257500,10386.153320
10000,8091.060000,8176.603516
15000,6842.916875,6666.051758
20000,5611.603125,5715.379395
25000,5217.630625,5204.868652
30000,4868.988125,4984.621582
35000,5071.327188,4906.698730
40000,4832.673437,4879.999023
45000,4981.174063,4870.512695
50000,4889.793750,4865.981934


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 34.39it/s]
/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 27.05it/s]
/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 28.50it/s]
/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.61it/s]
/opt/anacon


🏁 EXAMEN FINAL SUR LE TEST SET ISOLÉ...


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [6]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

print("⚡️ RÉCUPÉRATION ÉCLAIR DES RÉSULTATS...")

# 1. On demande au trainer de prédire sur le set de test (très rapide sur GPU M4)
predictions_output = trainer.predict(tokenized_test)

# 2. On extrait les chiffres
y_pred = predictions_output.predictions.flatten()
y_true = predictions_output.label_ids.flatten()

# 3. Les calculs magiques
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)
relative_rmse = (rmse / np.mean(y_true)) * 100

print("\n" + "="*40)
print(f"✅ R² Score         : {r2:.4f}")
print(f"📉 RMSE             : {rmse:.4f}")
print(f"🎯 RMSE Relatif     : {relative_rmse:.2f}%")
print("="*40)

if r2 > 0.8:
    print("💎 Ton modèle est une pépite.")
else:
    print("📈 C'est un bon début, l'élève progresse.")

⚡️ RÉCUPÉRATION ÉCLAIR DES RÉSULTATS...

✅ R² Score         : -0.0007
📉 RMSE             : 69.8008
🎯 RMSE Relatif     : 70.67%
📈 C'est un bon début, l'élève progresse.
